# doomformer: DOOM computes a neural network

[doomformer](https://github.com/mottopanikeiku/doomformer) compiles integer neural networks into DOOM maps. Numbers are
voodoo dolls (copies of the player that the game moves): an activation of 5 is five dolls. Each doll crosses lines whose
specials add a weight to a sector's light level, and those light levels are the sums. ViZDoom runs the maps headless;
Python writes the inputs, reads the results back and compares every value with a NumPy integer reference, with
tolerance 0.

*Runtime > Run all* runs everything. Cell 3 waits until you draw a digit and press Read. Colab's free tier has 2 vCPUs,
so cells 4 and 5 run 2 DOOM games at once. Cells 3 to 5 take one to two minutes each and print their own timings.

## 1. Install

doomformer v0.2.0 or later from GitHub, with ViZDoom 1.3.2. pip may suggest restarting the kernel; nothing here needs
it. `doomformer doctor` checks the parts. The first DOOM run (cell 2) builds DOOM's ACS script compiler (acc) from
source; that needs git, cmake and a C compiler, which Colab has.

In [ ]:
%pip install -q git+https://github.com/mottopanikeiku/doomformer
!doomformer doctor
assert globals().get("_exit_code", 0) == 0, "doomformer doctor found a problem (BAD lines above)"

## 2. Hello

A random integer MLP (4-6-5-3) compiled into one DOOM map. DOOM computes five random inputs; each output and each
hidden value it read back equals the integer reference.

In [ ]:
!doomformer hello
assert globals().get("_exit_code", 0) == 0, "doomformer hello failed"

## 3. Draw a digit

Draw one digit, large and centred, then press Read. Its 784 pixels go into an MLP trained on MNIST; the whole network,
down to picking the winning digit, is one DOOM map. One digit takes about 90 s.

Outside Colab the canvas cannot send the picture back, so the cell writes a shipped MNIST test digit to `digit.png`
instead. Any picture of one digit works: `doomformer mnist --image your.png`.

In [ ]:
JS = r"""
async function drawDigit() {
  const n = 280, box = document.createElement("div");
  const canvas = document.createElement("canvas");
  canvas.width = canvas.height = n;
  canvas.style.cssText = "border: 1px solid #888; touch-action: none; cursor: crosshair";
  const ctx = canvas.getContext("2d");
  const clear = () => { ctx.fillStyle = "white"; ctx.fillRect(0, 0, n, n); };
  clear();
  Object.assign(ctx, {strokeStyle: "black", lineWidth: 22, lineCap: "round", lineJoin: "round"});
  const at = e => { const r = canvas.getBoundingClientRect();
                    return [(e.clientX - r.left) * n / r.width, (e.clientY - r.top) * n / r.height]; };
  let down = false;
  canvas.onpointerdown = e => { down = true; canvas.setPointerCapture(e.pointerId);
                                ctx.beginPath(); ctx.moveTo(...at(e)); ctx.lineTo(...at(e)); ctx.stroke(); };
  canvas.onpointermove = e => { if (down) { ctx.lineTo(...at(e)); ctx.stroke(); } };
  canvas.onpointerup = canvas.onpointercancel = () => { down = false; };
  const button = text => { const b = document.createElement("button"); b.textContent = text; return b; };
  const clearButton = button("Clear"), readButton = button("Read");
  clearButton.onclick = clear;
  box.append(canvas, document.createElement("br"), clearButton, readButton);
  document.body.appendChild(box);
  if (window.google?.colab?.output) google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
  await new Promise(resolve => readButton.onclick = resolve);
  clearButton.disabled = readButton.disabled = true;
  return canvas.toDataURL("image/png");
}"""

import base64

try:
    from google.colab.output import eval_js
except ImportError:
    eval_js = None
if eval_js:
    from IPython.display import Javascript, display
    display(Javascript(JS))
    url = eval_js("drawDigit()")
    with open("digit.png", "wb") as f:
        f.write(base64.b64decode(url.split(",", 1)[1]))
else:
    import numpy as np
    from PIL import Image
    from doomformer import shipped
    d = np.load(shipped.data("mnist/digits.npz"))
    bits = np.unpackbits(d["bits"][0])[:784].reshape(28, 28)
    Image.fromarray(np.uint8(255 - 255 * bits)).resize((280, 280), Image.NEAREST).save("digit.png")
    print(f"not in Colab: wrote MNIST test digit #{d['test_index'][0]} (a {d['labels'][0]}) to digit.png")
!doomformer mnist --image digit.png
assert globals().get("_exit_code", 0) == 0, "doomformer mnist failed"

## 4. MNIST test digits

Two random MNIST test digits, read in 2 DOOM games at once.

In [ ]:
!doomformer mnist --digits 2
assert globals().get("_exit_code", 0) == 0, "doomformer mnist failed"

## 5. A transformer writes a DOOM level name

A small transformer trained on DOOM level names, with softmax attention: the dolls compute the exponentials and divide by
their sum. Its weights are split over 39 tile maps that run in one DOOM game per CPU (at most 8); Python copies the
values between tiles and picks each letter from the logits DOOM computed. `doomformer demo` without `--softmax` runs the
ReLU-attention model.

In [ ]:
!doomformer demo --softmax
assert globals().get("_exit_code", 0) == 0, "doomformer demo failed"

## 6. Film the dolls

`brain.record` runs XOR through a 2-2-1 net in DOOM and films it from above: every dot is a voodoo doll, white while it
walks. A pass is stamped EXACT only after its comparison with the integer reference passed.

In [ ]:
import itertools

import numpy as np
from IPython.display import Image, display

import doomformer as df

xor = df.MLP(weights=[np.array([[1, 1], [1, 1]]), np.array([[1, -1]])],    # hidden: OR, AND; output: OR - AND
             biases=[np.array([0, -1]), np.array([0])],
             thresholds=[np.array([1]), np.array([1])], signed_input=False)
inputs = [list(b) for b in itertools.product([0, 1], repeat=2)]
with df.compile(xor) as brain:
    rec = brain.record(inputs, "xor.gif", width=640)        # raises df.MismatchError on any mismatch
print(rec)
for x, y in zip(inputs, rec.results):
    print(f"x={x}  DOOM y={int(y[0])}  reference y={int(xor.reference(np.array(x))['y'][0])}")
display(Image(filename="xor.gif"))

## 7. More

* [README](https://github.com/mottopanikeiku/doomformer#readme): the other demos (`doomformer ask "what does iddqd do"`,
  `doomformer demo --nightmare`, `doomformer draw`) and the API.
* [How DOOM computes](https://github.com/mottopanikeiku/doomformer/blob/main/docs/HOW_IT_WORKS.md): weights, staircases,
  softmax attention, engine facts.
* [Examples](https://github.com/mottopanikeiku/doomformer/tree/main/examples): twelve one-file scripts.
* [The films](https://mottopanikeiku.github.io/doomformer/): DOOM computing, recorded.